# World Rupture — Remotion 3D Render on Google Colab (T4 GPU)

Run cells top to bottom. **Runtime → Change runtime type → T4 GPU → Save.**

- Uses **`--chrome-mode=chrome-for-testing`** (`--headless=new`), which is required on Linux for NVIDIA GPU access (Remotion's default `headless-shell` disables GPU drivers on Linux).
- Installs **EGL / GLES / Vulkan** vendors (`10_nvidia.json`) and verifies hardware WebGL via `npx remotion gpu` before rendering.
- Mounts Google Drive if available (saves chunks safely in `MyDrive/world_rupture/` across disconnects), with automatic fallback to `/content/AIVideo/out`.

## 1) Check GPU / CPU & Mount Google Drive (Optional)

In [ ]:
!nvidia-smi
!nproc

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/world_rupture'
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    print(f'Drive mounted. Outputs will be saved to: {OUTPUT_ROOT}')
except Exception as exc:
    OUTPUT_ROOT = '/content/AIVideo/out'
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    print(f'Drive not mounted ({exc}). Using local fallback: {OUTPUT_ROOT}')

## 2) Install Node.js 22 LTS, FFmpeg, Chromium & NVIDIA EGL/Vulkan GPU Libraries

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get update -qq
!sudo apt-get install -y nodejs ffmpeg libnss3 libdbus-1-3 libatk1.0-0 libgbm-dev libasound2 libxrandr2 libxkbcommon-dev libxfixes3 libxcomposite1 libxdamage1 libatk-bridge2.0-0 libpango-1.0-0 libcairo2 libcups2 libgtk-3-0 libx11-xcb1 libegl1 libgles2 libgl1 libglvnd0 libglx0 libvulkan1 mesa-vulkan-drivers vulkan-tools

import json, os
egl_vendor_dir = '/usr/share/glvnd/egl_vendor.d'
os.makedirs(egl_vendor_dir, exist_ok=True)
nvidia_egl_json = os.path.join(egl_vendor_dir, '10_nvidia.json')
if not os.path.exists(nvidia_egl_json):
    with open(nvidia_egl_json, 'w') as f:
        json.dump({'file_format_version': '1.0.0', 'ICD': {'library_path': 'libEGL_nvidia.so.0'}}, f, indent=2)
    print('Created NVIDIA EGL ICD config:', nvidia_egl_json)

!node -v && npm -v && ffmpeg -version | head -n 1

## 3) Clone Project from GitHub & Ensure Chrome-for-Testing Browser

In [ ]:
!rm -rf /content/AIVideo
!git clone https://github.com/abdullahmehmood1/AIVideo.git /content/AIVideo
%cd /content/AIVideo

# Optional Drive zip fallback (comment out git clone above if using a private Drive archive):
# !mkdir -p /content/AIVideo && unzip -q -o /content/drive/MyDrive/world_rupture/AIVideo.zip -d /content/AIVideo
# %cd /content/AIVideo

!npm ci || npm install
!npx remotion browser ensure --chrome-mode=chrome-for-testing
!npx remotion versions
!npx remotion compositions src/index.ts

## 4) Verify Hardware GPU Access (`remotion gpu`) & 60-Frame Benchmark
Checks `angle-egl`, `vulkan`, `egl`, and `angle` using `--chrome-mode=chrome-for-testing --hardware-acceleration=if-possible`, reports whether WebGL is hardware accelerated, and automatically selects the fastest GPU backend into `BEST_GL`.

In [ ]:
import os, subprocess, time
os.chdir('/content/AIVideo')

SCENE_FRAME_START = 0  # Change to benchmark any 60-frame range (e.g., 600 or 9000)
GL_CANDIDATES = ['angle-egl', 'vulkan', 'egl', 'angle']
hw_results = []
fallback_results = []

for gl in GL_CANDIDATES:
    gpu_info = subprocess.run([
        'npx', 'remotion', 'gpu', '--chrome-mode=chrome-for-testing',
        '--enable-multiprocess-on-linux', f'--gl={gl}'
    ], capture_output=True, text=True)
    gpu_out = (gpu_info.stdout + '\n' + gpu_info.stderr).strip()
    webgl_line = next((line for line in gpu_out.splitlines() if 'WebGL:' in line or 'WebGL2:' in line), 'WebGL: unknown')
    is_hw = 'Hardware accelerated' in gpu_out

    out = f'/content/bench_{gl}.mp4'
    if os.path.exists(out):
        os.remove(out)
    t0 = time.time()
    r = subprocess.run([
        'npx', 'remotion', 'render', 'src/index.ts', 'Main', out,
        f'--frames={SCENE_FRAME_START}-{SCENE_FRAME_START + 59}',
        '--chrome-mode=chrome-for-testing', '--hardware-acceleration=if-possible',
        '--codec=h264', '--muted', '--enable-multiprocess-on-linux',
        '--concurrency=2', f'--gl={gl}'
    ], capture_output=True, text=True)
    elapsed = round(time.time() - t0, 2)
    ok = (r.returncode == 0) and os.path.exists(out) and os.path.getsize(out) > 1000
    status = 'OK' if ok else 'FAILED'
    hw_tag = 'GPU' if is_hw else 'CPU/SW'
    print(f'{gl:10s} | {status:6s} | {hw_tag:6s} | {elapsed:6.2f}s/60f ({elapsed/60:.2f}s/f) | {webgl_line}')
    if ok:
        if is_hw:
            hw_results.append((elapsed, gl))
        else:
            fallback_results.append((elapsed, gl))
    else:
        print('  stderr:', r.stderr[-300:].strip())

if hw_results:
    hw_results.sort()
    BEST_GL = hw_results[0][1]
    est_min = round((hw_results[0][0] / 60.0) * 19350 / 60.0, 1)
    print(f'\nSelected hardware GPU backend BEST_GL = {BEST_GL!r} (~{est_min} min est. for 19,350 frames)')
elif fallback_results:
    fallback_results.sort()
    BEST_GL = fallback_results[0][1]
    print(f'\nNo hardware WebGL detected; using fastest working backend BEST_GL = {BEST_GL!r}')
else:
    BEST_GL = 'swangle'
    print('\nHardware candidates failed; falling back to software CPU mode BEST_GL = "swangle"')

## 5) Quick Preview Render (Test3D 60f or First 10 Seconds of Main)
Verify visuals & download a quick test clip before committing to the full chunked render.

In [ ]:
import os, subprocess
os.chdir('/content/AIVideo')
os.makedirs('out', exist_ok=True)

GL = globals().get('BEST_GL', 'angle-egl')
PREVIEW_MODE = 'Main10s'  # Options: 'Test3D' (60 frames) or 'Main10s' (frames 0-299)

if PREVIEW_MODE == 'Test3D':
    cmd = [
        'npx', 'remotion', 'render', 'src/index.ts', 'Test3D', 'out/preview.mp4',
        '--chrome-mode=chrome-for-testing', '--hardware-acceleration=if-possible',
        '--codec=h264', '--crf=18', '--muted', '--enable-multiprocess-on-linux',
        '--concurrency=2', f'--gl={GL}'
    ]
else:
    cmd = [
        'npx', 'remotion', 'render', 'src/index.ts', 'Main', 'out/preview.mp4',
        '--frames=0-299', '--chrome-mode=chrome-for-testing', '--hardware-acceleration=if-possible',
        '--codec=h264', '--crf=18', '--muted', '--enable-multiprocess-on-linux',
        '--concurrency=2', f'--gl={GL}'
    ]

print(f'Rendering preview ({PREVIEW_MODE}) with chrome-for-testing & --gl={GL}...')
subprocess.run(cmd, check=True)
print('Preview ready: /content/AIVideo/out/preview.mp4')

try:
    from google.colab import files
    files.download('out/preview.mp4')
except Exception as exc:
    print('Download skipped:', exc)

## 6) Render Full 19,350-Frame Documentary in 1,800-Frame Chunks (Resumable)
Saves muted `part_00.mp4` … `part_10.mp4` into `{OUTPUT_ROOT}/parts`. Already-finished chunks are automatically skipped on re-run.

In [ ]:
import os, shutil, subprocess, time
os.chdir('/content/AIVideo')

GL = globals().get('BEST_GL', 'angle-egl')
OUT_ROOT = globals().get('OUTPUT_ROOT', '/content/AIVideo/out')
PARTS_DIR = os.path.join(OUT_ROOT, 'parts')
os.makedirs(PARTS_DIR, exist_ok=True)

TOTAL = 19350
CHUNK = 1800

for idx, start in enumerate(range(0, TOTAL, CHUNK)):
    end = min(start + CHUNK - 1, TOTAL - 1)
    target = os.path.join(PARTS_DIR, f'part_{idx:02d}.mp4')
    if os.path.exists(target) and os.path.getsize(target) > 10000:
        print(f'[skip] {target} (frames {start}-{end})')
        continue

    tmp = '/content/part_tmp.mp4'
    if os.path.exists(tmp):
        os.remove(tmp)

    print(f'[render] part_{idx:02d}: frames {start}-{end} (chrome-for-testing, --gl={GL})')
    t0 = time.time()
    r = subprocess.run([
        'npx', 'remotion', 'render', 'src/index.ts', 'Main', tmp,
        f'--frames={start}-{end}', '--chrome-mode=chrome-for-testing', '--hardware-acceleration=if-possible',
        '--codec=h264', '--crf=18', '--muted',
        '--enable-multiprocess-on-linux', '--concurrency=2', f'--gl={GL}'
    ])
    if r.returncode != 0 or not os.path.exists(tmp):
        raise RuntimeError(f'Chunk part_{idx:02d} ({start}-{end}) failed with code {r.returncode}')

    shutil.copy2(tmp, target)
    print(f'[done] {target} in {round((time.time() - t0) / 60, 2)} min')

## 7) Concatenate Chunks & Mux Master Audio (if present)
Concatenates `part_*.mp4` into `video_noaudio.mp4`. If `master_audio.wav` (in `OUTPUT_ROOT`) or `public/audio/vo.wav` exists, muxes it cleanly into `final.mp4`; otherwise copies `video_noaudio.mp4` as `final.mp4`.

In [ ]:
import glob, os, shutil, subprocess

OUT_ROOT = globals().get('OUTPUT_ROOT', '/content/AIVideo/out')
PARTS_DIR = os.path.join(OUT_ROOT, 'parts')
parts = sorted(glob.glob(os.path.join(PARTS_DIR, 'part_*.mp4')))
if not parts:
    raise FileNotFoundError(f'No chunk files found in {PARTS_DIR}')

list_path = os.path.join(PARTS_DIR, 'list.txt')
with open(list_path, 'w') as f:
    for p in parts:
        f.write(f"file '{p}'\n")

video_noaudio = os.path.join(OUT_ROOT, 'video_noaudio.mp4')
subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', list_path,
    '-c', 'copy', video_noaudio
], check=True)

audio_candidates = [
    os.path.join(OUT_ROOT, 'master_audio.wav'),
    '/content/AIVideo/public/audio/vo.wav',
]
audio_path = next((a for a in audio_candidates if os.path.exists(a) and os.path.getsize(a) > 1000), None)
final_mp4 = os.path.join(OUT_ROOT, 'final.mp4')

if audio_path:
    print(f'Muxing audio from {audio_path} into {final_mp4}...')
    subprocess.run([
        'ffmpeg', '-y', '-i', video_noaudio, '-i', audio_path,
        '-c:v', 'copy', '-c:a', 'aac', '-b:a', '320k', '-shortest', final_mp4
    ], check=True)
else:
    shutil.copy2(video_noaudio, final_mp4)
    print(f'No master audio found yet; saved video-only master to {final_mp4}')

size_mb = os.path.getsize(final_mp4) / (1024 * 1024)
print(f'Final file ready: {final_mp4} ({size_mb:.1f} MB)')

## Optional: Generate Word-Level `captions.json` with Faster-Whisper on T4 GPU
Upload `vo.wav` to `OUTPUT_ROOT` (e.g. `MyDrive/world_rupture/vo.wav`), run this cell, then place the generated `captions.json` into `src/data/`.

In [ ]:
!pip -q install faster-whisper

import json, os
from faster_whisper import WhisperModel

OUT_ROOT = globals().get('OUTPUT_ROOT', '/content/AIVideo/out')
vo_path = os.path.join(OUT_ROOT, 'vo.wav')
if not os.path.exists(vo_path):
    vo_path = '/content/AIVideo/public/audio/vo.wav'

model = WhisperModel('large-v3', device='cuda', compute_type='float16')
segments, info = model.transcribe(vo_path, word_timestamps=True, language='en')

caps = []
for seg in segments:
    for i, w in enumerate(seg.words):
        # Keep leading space so Remotion createTikTokStyleCaptions splits silences reliably
        word_text = w.word if (len(caps) == 0 or w.word.startswith(' ')) else f' {w.word}'
        caps.append({
            'text': word_text,
            'startMs': int(w.start * 1000),
            'endMs': int(w.end * 1000),
            'timestampMs': int((w.start + w.end) * 500),
            'confidence': float(w.probability),
        })

out_json = os.path.join(OUT_ROOT, 'captions.json')
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(caps, f, indent=2)
print(f'Saved {len(caps)} word captions to {out_json}')

## Optional: Mix Voiceover + Music with Automatic Ducking
Place `vo.wav` and `music.mp3` in `OUTPUT_ROOT` (or `public/audio/`) to produce `master_audio.wav`.

In [ ]:
import os, subprocess

OUT_ROOT = globals().get('OUTPUT_ROOT', '/content/AIVideo/out')
vo_in = os.path.join(OUT_ROOT, 'vo.wav') if os.path.exists(os.path.join(OUT_ROOT, 'vo.wav')) else '/content/AIVideo/public/audio/vo.wav'
music_in = os.path.join(OUT_ROOT, 'music.mp3') if os.path.exists(os.path.join(OUT_ROOT, 'music.mp3')) else '/content/AIVideo/public/audio/music.mp3'
master_out = os.path.join(OUT_ROOT, 'master_audio.wav')

subprocess.run([
    'ffmpeg', '-y', '-i', vo_in, '-i', music_in,
    '-filter_complex',
    '[1:a]volume=0.35[m];[m][0:a]sidechaincompress=threshold=0.03:ratio=8:attack=20:release=400[duck];[0:a][duck]amix=inputs=2:duration=first:normalize=0[a]',
    '-map', '[a]', '-ar', '48000', master_out
], check=True)
print('Saved mixed audio:', master_out)